## KPI Resolver
Original script created by j.rosenbauer@celonis.com

This python scripts extracts a given Knowledge Model from your Celonis team and will resolve all KPIs down to the data table fields used (as referenced to in the connector).
These fields are then extracted extracted and shown next to the resolved query.

### Variables required
* team_url: URL of your desired Celonis team
* user_api_token: your user key with which you want to run the script
* space_key: key of the space your desired Knowledge Model resides in

### Required user input
* Package ID: found by querying the space referenced by the key above (note: not key or name, but id - using key or name can cause errors)
* Knowledge Model ID: found by querying the package referenced by the ID above (note: not key or name, but id - using key or name can cause errors)

### Known issues
* If a KPI is used as input-parameter for a parameterized KPI, the resolution will break due to not finding the 'correct' exit to recursion

### Output
* Excel or CSV with ID, Name, Description, Original PQL, Resolved PQL, KPI-type, Aggregation Level (the latter two are depending on KPI-Nomenclature)

In [1]:
team_url = '' # Your team's URL
user_api_token = '' # Your API token (go to your profile in EMS and generate a key there if not available)
space_key = '' # Key of the space the desired packages are in

In [2]:
import pycelonis
import re
import yaml
import pandas as pd
import numpy as np

c:\Python310\lib\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Connecting to Team
from pycelonis import get_celonis
login_target = {
    "base_url": team_url,
    "api_token": user_api_token,
    "key_type": "USER_KEY"
}
celonis = get_celonis(**login_target, timeout=1200) # uncomment this to use login details.

#### The cell below provides all package IDs from the referenced space. Pick one and paste it into the input window opened by the next cell.

In [4]:
# Get relevant objects
space = celonis.studio.get_space(space_key)
print(space)
packages = space.get_packages()
print(packages)

permissions=['CREATE_PACKAGE', 'DELETE_SPACE', 'USE', '$ACCESS_CHILD', 'CREATE_SPACE', 'DELETE_ALL_PACKAGES', 'USE_ALL_PACKAGES', 'DELETE_PACKAGE', 'EDIT_PACKAGE', 'EDIT_ALL_PACKAGES', 'EDIT_ALL_SPACES', 'MANAGE_PERMISSIONS', 'EDIT_SPACE', 'DELETE_ALL_SPACES', 'USE_PACKAGE'] id='3a309d92-662d-0eab-5b0d-779b02a17c1d' name='a. Productive' icon_reference='store' creation_date=datetime.datetime(2021, 8, 23, 12, 7, 35, 440000, tzinfo=datetime.timezone.utc) change_date=datetime.datetime(2021, 8, 23, 12, 7, 35, 440000, tzinfo=datetime.timezone.utc) object_id='3a309d92-662d-0eab-5b0d-779b02a17c1d' client=<pycelonis_core.client.client.Client object at 0x0000027B2D6D8FA0>
[
	Package(id='cc38420e-2573-4647-bea7-2231d4200817', key='im-starter-kit', name='IM Starter Kit', root_node_key='im-starter-kit', space_id='3a309d92-662d-0eab-5b0d-779b02a17c1d'),
	Package(id='87dacc78-fc77-4d2f-baf1-148288b90e9e', key='im-control-center', name='IM Control Center', root_node_key='im-control-center', space_id='

In [5]:
# Enter below the id of the desired package from the output of the cell above
package = space.get_package(input())
package

Package(id='cc38420e-2573-4647-bea7-2231d4200817', key='im-starter-kit', name='IM Starter Kit', root_node_key='im-starter-kit', space_id='3a309d92-662d-0eab-5b0d-779b02a17c1d')

#### The cell below provides all knowledge model IDs from the referenced package. Pick one and paste it into the input window opened by the next cell.

In [6]:
package.get_content_nodes()
km = package.get_knowledge_models()
km

[
	KnowledgeModel(id='af9514d3-4294-4fc4-8336-92bf2ed342fd', key='-delete-im-master-knowledge-model-for-resolver', name='[DELETE]IM Master Knowledge Model For Resolver', root_node_key='im-starter-kit', space_id='3a309d92-662d-0eab-5b0d-779b02a17c1d'),
	KnowledgeModel(id='337d1b2d-6fe1-4bba-a5a6-e630b95cbb1f', key='im-master-knowledge-model', name='IM Master Knowledge Model', root_node_key='im-starter-kit', space_id='3a309d92-662d-0eab-5b0d-779b02a17c1d')
]

In [7]:
# Check to see if the right Knowledge Model is referenced
km = package.get_knowledge_model(input())
km.serialized_content

'kind: BASE\nmetadata:\n  key: -delete-im-master-knowledge-model-for-resolver\n  displayName: "[DELETE]IM Master Knowledge Model For Resolver"\ndataModelId: ${{inventory-management-data-model}}\nrecords:\n  - id: MATERIAL_PLANT\n    displayName: Material Plant\n    identifier:\n      id: MATERIAL_PLANT_ID\n      displayName: Material Number - Plant\n      pql: |\n        "MARC"."MANDT" || \' - \' || "MARC"."MATNR" || \' - \' || "MARC"."WERKS"\n    pql: MARC\n    attributes:\n      - id: CLIENT\n        displayName: SAP Client\n        pql: |\n          "MARC"."MANDT"\n      - id: MATERIAL_NUMBER_UNFORMATTED\n        displayName: Material Number w/ Leading Zeroes\n        pql: |\n          "MARC"."MATNR"\n      - id: MATERIAL_NUMBER\n        displayName: Material Number\n        pql: |\n          (REPLACE(LTRIM(REPLACE("MARC"."MATNR",\'0\',\' \')),\' \',\'0\'))\n        columnType: string\n      - id: MATERIAL_NAME\n        displayName: Material Name\n        pql: |\n          "MARC"."M

### Start Processing the Knowledge Model
* Extract content with variables already resolved
* Create a list of KPI dicts
* Recursively resolve KPIs
* Transfer to dataframe, add additional context, write to Excel or CSV

Note: Excel has a limit of ~32,700 characters in a cell. KPIs which are longer than that will be cut off. In that case you should export to CSV.

In [8]:
# Extract KM content
kpi_list = km.get_content(
    with_variable_replacement=True,
    with_autogenerated_data_model_data=False,
    with_default_values=False,
    validate_pql=False,
    with_unknown_variables_validation=False,
).kpis

In [9]:
# Iterate across output and generate dict with references and keywords
references = list()

for kpi in kpi_list:
    references.append({'id' : kpi.id, 'name' : kpi.display_name, 'description' : kpi.description, 'pql' : kpi.pql})
    
print(len(references))


469


#### Logic to Resolve PQL and extract fields

In [10]:
re_operator = re.compile(r'KPI\(')
re_id = re.compile(r'KPI\(\s*(.*?)\s*\)')

def expand_kpi(query: str):
    """Recursively replaces any occurrences of KPI operator with its PQL."""
    # Check if there is still a KPI-operator in the query

    if re_operator.search(query):
        print('Operator found')
        # print(query)
        # Find ID of referenced KPI

        # Debugging: if query encounters Nonetype, print the query
        if re_id.search(query) is None: 
            print('Nonetype encountered')
            print(query)
        
        kpi_id = re_id.search(query).group(1)
        #print(kpi_id)          

        kpi_replace = kpi_id    # Store full operator call for replacement at the end of the function
        #print(kpi_replace)

        # There are edge cases where the KPI-operator is not dropped correctly, resolve with this statement
        if kpi_id[0:4] == 'KPI(':
            print('Engage failsafe to drop PQL-operator properly')
            kpi_id = kpi_id[4:]

        # Handle parameterized KPIs
        kpi_id = kpi_id.split(',')[0]
        print(kpi_id)

        # Find the right KPI, then break immediately
        for element in references:
            if element['id'] == kpi_id:
                kpi_pql = element['pql']
                print('PQL statement found')
                break
        else:
            kpi_pql = None
            print('No PQL statement found')
        
        query = re.sub(re.compile(r'KPI\(\s*' + kpi_replace + r'\s*\)'), kpi_pql, query)
        return expand_kpi(query)
    return query

def extract_fields(query: str):
    breakdown = query.split('"')
    result = ''
    counter = 0
    while counter < len(breakdown):
        if breakdown[counter] == '.':
            field = breakdown[counter - 1] + breakdown[counter] + breakdown[counter + 1]
            if (field in result) == False: 
                # print(breakdown[counter - 1] + breakdown[counter] + breakdown[counter + 1])
                result = result + field + ', \n'
        counter = counter + 1
    
    result = result [:-3]   # Remove last comma and new line 
    #print(result)
    return result


In [11]:
for element in references:
    print('Resolving KPI ' + element['id'])
    element['resolved_pql'] = expand_kpi(element['pql'])

Resolving KPI KPI_PLANT_COUNT_TABLE_MARC
Resolving KPI KPI_PLANT_COUNT_DISTINCT_MATERIAL_MOVEMENTS
Resolving KPI KPI_PLANT_COUNT_DISTINCT_MATERIAL_ACTIVE_STOCK
Operator found
KPI_MATERIAL_INDICATOR_INVENTORY_CLASSIFICATION
PQL statement found
Operator found
FORMULA_MATERIAL_INDICATOR_STOCK_OUT
PQL statement found
Operator found
KPI_MATERIAL_PULAST_INVENTORY_ON_HAND_QUANTITY
PQL statement found
Operator found
KPI_MATERIAL_PULAST_CONSIGNMENT_STOCK_QUANTITY
PQL statement found
Operator found
FORMULA_MATERIAL_INDICATOR_STOCK_UNDER
PQL statement found
Operator found
KPI_MATERIAL_PULAST_INVENTORY_ON_HAND_QUANTITY
PQL statement found
Operator found
KPI_MATERIAL_PULAST_CONSIGNMENT_STOCK_QUANTITY
PQL statement found
Operator found
FORMULA_MATERIAL_INDICATOR_STOCK_OBSOLETE
PQL statement found
Operator found
KPI_MATERIAL_PUSUM_EXCESS_INVENTORY_RATE
PQL statement found
Operator found
KPI_MATERIAL_PUSUM_ACTIVE_STOCK_VALUE
PQL statement found
Operator found
KPI_MATERIAL_PULAST_INVENTORY_ON_HAND_VALU

In [12]:
for element in references:
    print('Extracting fields from ' + element['id'])
    element['fields'] = extract_fields(element['resolved_pql'])

Extracting fields from KPI_PLANT_COUNT_TABLE_MARC
Extracting fields from KPI_PLANT_COUNT_DISTINCT_MATERIAL_MOVEMENTS
Extracting fields from KPI_PLANT_COUNT_DISTINCT_MATERIAL_ACTIVE_STOCK
Extracting fields from KPI_PLANT_COUNT_DISTINCT_MATERIAL_CONSIDERED_EXCESS_STOCK
Extracting fields from KPI_PLANT_COUNT_DISTINCT_MATERIAL_EXCLUDED_MRP_TYPE
Extracting fields from KPI_PLANT_COUNT_DISTINCT_MATERIAL_EXCLUDED_PLANNING_STRATEGY
Extracting fields from KPI_MATERIAL_CALC_PERCENTAGE_SELECTED_ITEMS
Extracting fields from KPI_MATERIAL_CALC_PERCENTAGE_SELECTED_ITEMS_EXCESS_STOCK
Extracting fields from KPI_MATERIAL_PUCOUNT_TASKS_OPEN
Extracting fields from FORMULA_PURCHASE_ORDER_CURRENCYCONVERT
Extracting fields from FORMULA_STOCKHISTORY_CURRENCYCONVERT_INVENTORY
Extracting fields from FORMULA_STOCKHISTORY_CURRENCYCONVERT_CONSUMPTION
Extracting fields from FORMULA_STOCKHISTORY_CURRENCYCONVERT_REPLENISHMENT
Extracting fields from FORMULA_STOCKHISTORY_CURRENCYCONVERT_MOVING_AVERAGE_PRICE
Extracting f

In [13]:
# Write resolved output to df
km_resolved = pd.DataFrame(references)
# km_resolved

In [14]:
# Add additional context and details
km_resolved['category'] = km_resolved['id'].str.split('_').str[0]
km_resolved['aggregation'] = km_resolved['id'].str.split('_').str[1]
km_resolved['length'] = km_resolved['resolved_pql'].str.len()
km_resolved

,id,name,desription,pql,resolved_pql,fields,category,aggregation,length
0,KPI_PLANT_COUNT_TABLE_MARC,No. of Materials,This KPI is the number of unique combinations ...,"COUNT_TABLE(""MARC"")\n","COUNT_TABLE(""MARC"")\n",,KPI,PLANT,20
1,KPI_PLANT_COUNT_DISTINCT_MATERIAL_MOVEMENTS,No. of Movements,This KPI is the number of unique combinations ...,"COUNT(DISTINCT ""_CEL_IM_ACTIVITIES"".""MANDT"" ||...","COUNT(DISTINCT ""_CEL_IM_ACTIVITIES"".""MANDT"" ||...","_CEL_IM_ACTIVITIES.MANDT, \n_CEL_IM_ACTIVITIES...",KPI,PLANT,108
2,KPI_PLANT_COUNT_DISTINCT_MATERIAL_ACTIVE_STOCK,No. of Materials w/ Active Stock,This KPI is the number of unique combinations ...,COUNT(DISTINCT \n CASE \n WHEN KPI(KPI_MAT...,COUNT(DISTINCT \n CASE \n WHEN CASE\n WHE...,"STOCK_HISTORY.VALU_LBKUM_CON, \nSTOCK_HISTORY....",KPI,PLANT,67132
3,KPI_PLANT_COUNT_DISTINCT_MATERIAL_CONSIDERED_E...,No. of Materials Considered,This KPI is the number of unique combinations ...,COUNT(DISTINCT \n CASE \n WHEN KPI(KPI_MAT...,COUNT(DISTINCT \n CASE \n WHEN CASE\n WHE...,"STOCK_HISTORY.VALU_LBKUM_CON, \nSTOCK_HISTORY....",KPI,PLANT,134155
4,KPI_PLANT_COUNT_DISTINCT_MATERIAL_EXCLUDED_MRP...,No. of Materials Excluded through MRP Type,This KPI is the number of unique combinations ...,"COUNT(DISTINCT \n CASE \n WHEN ""MARC"".""DIS...","COUNT(DISTINCT \n CASE \n WHEN ""MARC"".""DIS...","MARC.DISMM, \nMARC.MANDT, \nMARC.MATNR, \nMARC...",KPI,PLANT,167
...,...,...,...,...,...,...,...,...,...
464,DISPLAY_CONSTANT_PARAMETER_HIGH_PERFORMANCE_DA...,High Performance,None,30,30,,DISPLAY,CONSTANT,2
465,DISPLAY_CONSTANT_PARAMETER_TARGET_SERVICE_LEVEL,Target,None,0.9,0.9,,DISPLAY,CONSTANT,3
466,DISPLAY_CONSTANT_PARAMETER_BENCHMARK_SERVICE_L...,Benchmark,None,0.85,0.85,,DISPLAY,CONSTANT,4
467,DISPLAY_CONSTANT_PARAMETER_LOW_PERFORMANCE_SER...,Low Performance,None,0.5,0.5,,DISPLAY,CONSTANT,3


#### Write results to Excel and/ or CSV

In [15]:
# Export to Excel
def km_result_to_excel(result, file_name:str):
    result.to_excel(file_name)  
    print("Wrote resolved PQL to file {}".format(file_name))

# Export to CSV
def km_result_to_csv(result, file_name:str):
    result.to_csv(file_name, sep=';')  
    print("Wrote resolved PQL to file {}".format(file_name))

km_result_to_excel(km_resolved, "knowledge_model_resolved.xlsx")
km_result_to_csv(km_resolved, "knowledge_model_resolved.csv")

Wrote resolved PQL to file knowledge_model_resolved.xlsx
Wrote resolved PQL to file knowledge_model_resolved.csv
